In [1]:
import heapq
import math


class Node:
    """Represents a position/node on the grid for A* pathfinding."""

    def __init__(self, position, parent=None):
        self.position = position  # Tuple (row, col)
        self.parent = parent

        self.g = 0  # Cost from start node to current node
        self.h = 0  # Estimated cost from current node to end node (heuristic)
        self.f = 0  # Total cost (f = g + h)

    def __lt__(self, other):
        # Comparison operator for the min-priority queue (heapq)
        return self.f < other.f


def heuristic(p1, p2, method="manhattan"):
    """Calculate heuristic distance between two points (row1, col1) and (row2, col2)."""
    (r1, c1), (r2, c2) = p1, p2
    if method == "euclidean":
        return math.sqrt((r1 - r2) ** 2 + (c1 - c2) ** 2)
    elif method == "manhattan":
        return abs(r1 - r2) + abs(c1 - c2)


def astar(grid, start, end, allow_diagonal=False):
    """Computes the shortest path on a 2D grid using the A* search algorithm.

    Parameters:
        grid (list of list of int): 0 for walkable cell, 1 for obstacle/wall.
        start (tuple): (row, col) starting coordinate.
        end (tuple): (row, col) target coordinate.
        allow_diagonal (bool): Whether 8-directional movement is allowed.

    Returns:
        list of tuple: Path from start to end as [(r1, c1), (r2, c2), ...], or None if blocked.
    """
    rows, cols = len(grid), len(grid[0])

    start_node = Node(start)
    end_node = Node(end)

    # Priority Queue for Open Set (Nodes to be evaluated)
    open_heap = []
    heapq.heappush(open_heap, start_node)

    # Dictionaries to track lowest g-costs and visited locations
    open_dict = {start: start_node}
    closed_set = set()

    # Movement vectors
    if allow_diagonal:
        # 8-directional movement (orthogonal cost = 1.0, diagonal cost = sqrt(2))
        directions = [
            (0, 1, 1.0),
            (1, 0, 1.0),
            (0, -1, 1.0),
            (-1, 0, 1.0),
            (1, 1, 1.414),
            (1, -1, 1.414),
            (-1, 1, 1.414),
            (-1, -1, 1.414),
        ]
        h_type = "euclidean"
    else:
        # 4-directional orthogonal movement
        directions = [(0, 1, 1.0), (1, 0, 1.0), (0, -1, 1.0), (-1, 0, 1.0)]
        h_type = "manhattan"

    while open_heap:
        # Pop node with lowest f-score
        current_node = heapq.heappop(open_heap)
        current_pos = current_node.position

        # Remove from open_dict and mark visited
        open_dict.pop(current_pos, None)
        closed_set.add(current_pos)

        # Target reached -> Reconstruct path
        if current_pos == end:
            path = []
            curr = current_node
            while curr:
                path.append(curr.position)
                curr = curr.parent
            return path[::-1]  # Return reversed path (start to end)

        # Explore neighbors
        for dr, dc, move_cost in directions:
            neighbor_pos = (current_pos[0] + dr, current_pos[1] + dc)
            nr, nc = neighbor_pos

            # 1. Bounds check
            if not (0 <= nr < rows and 0 <= nc < cols):
                continue

            # 2. Obstacle check
            if grid[nr][nc] == 1:
                continue

            # 3. Already evaluated check
            if neighbor_pos in closed_set:
                continue

            tentative_g = current_node.g + move_cost

            # If neighbor is already in open set with a lower/equal g cost, skip
            if (
                neighbor_pos in open_dict
                and tentative_g >= open_dict[neighbor_pos].g
            ):
                continue

            # Create or update neighbor node
            neighbor_node = Node(neighbor_pos, current_node)
            neighbor_node.g = tentative_g
            neighbor_node.h = heuristic(neighbor_pos, end, method=h_type)
            neighbor_node.f = neighbor_node.g + neighbor_node.h

            # Add to open set
            open_dict[neighbor_pos] = neighbor_node
            heapq.heappush(open_heap, neighbor_node)

    return None  # No path found


# ==========================================
# Example Usage & Visualization Helper
# ==========================================
def print_grid(grid, path=[], start=None, end=None):
    """Utility to print the grid with the calculated path."""
    grid_copy = [
        ["█" if cell == 1 else "·" for cell in row] for row in grid
    ]

    for r, c in path:
        grid_copy[r][c] = "o"

    if start:
        grid_copy[start[0]][start[1]] = "S"
    if end:
        grid_copy[end[0]][end[1]] = "E"

    for row in grid_copy:
        print(" ".join(row))


if __name__ == "__main__":
    # 0 = Open Cell, 1 = Wall/Obstacle
    grid = [
        [0, 0, 0, 0, 0, 0, 0],
        [0, 1, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 1, 0],
        [0, 1, 1, 1, 0, 1, 0],
        [0, 0, 0, 1, 0, 0, 0],
    ]

    start_pos = (0, 0)
    end_pos = (4, 6)

    path = astar(grid, start_pos, end_pos, allow_diagonal=False)

    print("Path Found:")
    print(path)
    print("\nVisualized Grid (S=Start, E=End, o=Path, █=Obstacle):")
    print_grid(grid, path=path or [], start=start_pos, end=end_pos)